# 1. Analysis - Cash Flow Statement

## Notebook Summary

This notebook analyzes a single company's cash flow statement history from Financial Modeling Prep using the local `TICKER` parameter near the top of this notebook. It is organized into data loading and preparation, annual and quarterly summary snapshots, core cash flow trend charts, and cash-conversion, working-capital, and capital-allocation diagnostics.

Run the notebook from top to bottom after updating the local `TICKER` parameter near the top of this notebook so the helper functions, annual and quarterly datasets, and all downstream tables and charts stay in sync.

## Data Loading and Preparation

These cells initialize the FMP helpers, normalize the ticker, load annual and quarterly cash flow statement history, and calculate the derived cash-generation, working-capital, capital-allocation, and TTM fields used throughout the notebook.

They also build the base annual and quarterly summary tables that the rest of the analysis depends on.

In [ ]:
# 2. Import libraries
from pathlib import Path
import sys

import pandas as pd

In [ ]:
# 2b. Set notebook parameters and project setup
from pathlib import Path
import sys

import pandas as pd

financial_statement_params = {
    "ticker_str": "SOXL",
    "annual_limit": 20,
    "quarterly_limit": 40,
}

TICKER = financial_statement_params["ticker_str"]
ANNUAL_LIMIT = financial_statement_params["annual_limit"]
QUARTERLY_LIMIT = financial_statement_params["quarterly_limit"]

# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()

from Quantapp.data import get_cash_flow_history


TICKER

In [ ]:
# 2c. Cash flow display helpers
def format_metric_value(value: float, style: str) -> str:
    if pd.isna(value):
        return "-"
    if style == "currency":
        return f"{value:,.0f}"
    if style == "percent":
        return f"{value:.2%}"
    if style == "ratio":
        return f"{value:,.2f}"
    return f"{value}"

In [ ]:
# 2d. Data retrieval is handled by Quantapp.data

In [ ]:
# 3. Load cash flow statement history through Quantapp.data
cash_flow_history = get_cash_flow_history(
    TICKER,
    annual_limit=ANNUAL_LIMIT,
    quarterly_limit=QUARTERLY_LIMIT,
)

SYMBOL = cash_flow_history.symbol
company_name = cash_flow_history.company_name
chart_label = cash_flow_history.chart_label
annual_cash = cash_flow_history.annual
quarterly_cash = cash_flow_history.quarterly

annual_cash.tail()

In [ ]:
# 3b. Review annual cash flow analytics
annual_cash.tail()

In [ ]:
# 4. Review quarterly cash flow statement history
quarterly_cash.tail()

In [ ]:
# 4b. Review quarterly cash flow analytics
quarterly_cash.tail()

In [ ]:
# 5. Build annual summary stats
from IPython.display import display

latest_annual = annual_cash.iloc[-1]
annual_summary = pd.DataFrame(
    [
        {"metric": "Latest annual net income", "value": latest_annual["netIncome"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual operating cash flow", "value": latest_annual["operatingCashFlow"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual capital expenditure", "value": latest_annual["capitalExpenditure"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual free cash flow", "value": latest_annual["freeCashFlow"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual shareholder returns", "value": latest_annual["shareholderReturns"], "style": "currency", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual CFO to net income", "value": latest_annual["cfoToNetIncome"], "style": "ratio", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual capex to CFO", "value": latest_annual["capexToCfo"], "style": "percent", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual FCF to CFO", "value": latest_annual["fcfToCfo"], "style": "percent", "asOf": latest_annual["date"].date()},
        {"metric": "Latest annual shareholder returns to FCF", "value": latest_annual["shareholderReturnsToFcf"], "style": "percent", "asOf": latest_annual["date"].date()},
    ]
)
annual_summary_display = annual_summary.copy()
annual_summary_display["value"] = [format_metric_value(value, style) for value, style in zip(annual_summary_display["value"], annual_summary_display["style"])]
display(annual_summary_display.loc[:, ["metric", "value", "asOf"]])

In [ ]:
# 6. Build quarterly summary stats
from IPython.display import display

latest_quarter = quarterly_cash.iloc[-1]
latest_ttm = quarterly_cash.dropna(subset=["ttmOperatingCashFlow"]).iloc[-1]
quarterly_summary = pd.DataFrame(
    [
        {"metric": "Latest quarterly net income", "value": latest_quarter["netIncome"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly operating cash flow", "value": latest_quarter["operatingCashFlow"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest quarterly free cash flow", "value": latest_quarter["freeCashFlow"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest TTM operating cash flow", "value": latest_ttm["ttmOperatingCashFlow"], "style": "currency", "asOf": latest_ttm["date"].date()},
        {"metric": "Latest TTM free cash flow", "value": latest_ttm["ttmFreeCashFlow"], "style": "currency", "asOf": latest_ttm["date"].date()},
        {"metric": "Latest quarterly CFO to net income", "value": latest_quarter["cfoToNetIncome"], "style": "ratio", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest TTM CFO to net income", "value": latest_ttm["ttmCfoToNetIncome"], "style": "ratio", "asOf": latest_ttm["date"].date()},
        {"metric": "Latest quarterly working capital change", "value": latest_quarter["changeInWorkingCapital"], "style": "currency", "asOf": latest_quarter["date"].date()},
        {"metric": "Latest TTM shareholder returns", "value": latest_ttm["ttmShareholderReturns"], "style": "currency", "asOf": latest_ttm["date"].date()},
    ]
)
quarterly_summary_display = quarterly_summary.copy()
quarterly_summary_display["value"] = [format_metric_value(value, style) for value, style in zip(quarterly_summary_display["value"], quarterly_summary_display["style"])]
display(quarterly_summary_display.loc[:, ["metric", "value", "asOf"]])

## Trend and Cash Generation Charts

These cells move from summary tables into visualization. They cover annual and quarterly cash-generation trends, working-capital and non-cash drivers, investing and financing flows, and the growth profile of net income, operating cash flow, and free cash flow.

In [ ]:
# 7. Plot annual cash flow trends
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_cash_flow_trends

annual_cash_fig = plot_cash_flow_trends(
    annual_cash,
    chart_label=chart_label,
    period_label="annual",
)
annual_cash_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 8. Plot quarterly cash flow trends
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_cash_flow_trends

quarterly_cash_fig = plot_cash_flow_trends(
    quarterly_cash,
    chart_label=chart_label,
    period_label="quarterly",
)
quarterly_cash_fig.show(config={"responsive": True, "displaylogo": False})

## Cash Conversion and Capital Allocation

These cells summarize whether accounting earnings are turning into cash, how much operating cash is being reinvested, and how aggressively management is returning cash through buybacks and dividends relative to free cash flow.

In [ ]:
# 9. Build cash conversion tables
from IPython.display import Markdown, display

annual_conversion_frame = annual_cash.tail(min(len(annual_cash), 6)).copy()
annual_conversion_labels = annual_conversion_frame["calendarYear"].astype("Int64").astype(str)
annual_conversion_table = pd.DataFrame(
    [
        annual_conversion_frame["cfoToNetIncome"].reset_index(drop=True).tolist(),
        annual_conversion_frame["capexToCfo"].reset_index(drop=True).tolist(),
        annual_conversion_frame["fcfToCfo"].reset_index(drop=True).tolist(),
        annual_conversion_frame["stockCompToCfo"].reset_index(drop=True).tolist(),
        annual_conversion_frame["workingCapitalDragToCfo"].reset_index(drop=True).tolist(),
        annual_conversion_frame["buybacksToFcf"].reset_index(drop=True).tolist(),
        annual_conversion_frame["dividendsToFcf"].reset_index(drop=True).tolist(),
        annual_conversion_frame["shareholderReturnsToFcf"].reset_index(drop=True).tolist(),
    ],
    index=[
        "CFO to net income",
        "Capex to CFO",
        "FCF to CFO",
        "Stock comp to CFO",
        "Working-capital drag to CFO",
        "Buybacks to FCF",
        "Dividends to FCF",
        "Shareholder returns to FCF",
    ],
    columns=annual_conversion_labels.tolist(),
)

quarterly_conversion_frame = quarterly_cash.dropna(subset=["ttmOperatingCashFlow"]).tail(min(len(quarterly_cash.dropna(subset=["ttmOperatingCashFlow"])), 8)).copy()
quarterly_conversion_labels = (
    quarterly_conversion_frame["date"].dt.year.astype(str)
    + " "
    + quarterly_conversion_frame["quarterLabel"].astype(str)
    + " ("
    + quarterly_conversion_frame["date"].dt.strftime("%b %Y")
    + ")"
)
quarterly_conversion_table = pd.DataFrame(
    [
        quarterly_conversion_frame["ttmCfoToNetIncome"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmCapexToCfo"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmFcfToCfo"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["stockCompToCfo"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["workingCapitalDragToCfo"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmBuybacksToFcf"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmDividendsToFcf"].reset_index(drop=True).tolist(),
        quarterly_conversion_frame["ttmShareholderReturnsToFcf"].reset_index(drop=True).tolist(),
    ],
    index=[
        "TTM CFO to net income",
        "TTM capex to CFO",
        "TTM FCF to CFO",
        "Quarterly stock comp to CFO",
        "Quarterly working-capital drag to CFO",
        "TTM buybacks to FCF",
        "TTM dividends to FCF",
        "TTM shareholder returns to FCF",
    ],
    columns=quarterly_conversion_labels.tolist(),
)

display(Markdown(f"### {SYMBOL} annual cash conversion and capital allocation"))
display(annual_conversion_table.round(2))
display(Markdown(f"### {SYMBOL} quarterly and TTM cash conversion and capital allocation"))
display(quarterly_conversion_table.round(2))

In [ ]:
# 10. Plot cash conversion and capital allocation
from Quantapp.visualization.views.single_asset_profile.valuation.fundamentals import plot_cash_conversion_capital_allocation

cash_conversion_fig = plot_cash_conversion_capital_allocation(
    annual_cash,
    quarterly_cash,
    chart_label=chart_label,
)
cash_conversion_fig.show(config={"responsive": True, "displaylogo": False})